<a href="https://colab.research.google.com/github/sumin031130/ESAA/blob/main/YB_1113_%E1%84%8B%E1%85%A7%E1%86%AB%E1%84%89%E1%85%B3%E1%86%B8%E1%84%86%E1%85%AE%E1%86%AB%E1%84%8C%E1%85%A6_%E1%84%92%E1%85%AC%E1%84%80%E1%85%B11_%E1%84%91%E1%85%AE%E1%86%AF%E1%84%8B%E1%85%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. 단순 선형 회귀 분석**
- 전복의 나이를 예측하는 선형회귀모델을 생성하세요.
- 전복의 ‘성별’, ‘키’, ‘지름’, ‘높이’, ‘전체무게’, ‘몸통무게’, ‘내장무게’, ‘껍질무게’를 이용해 ‘껍질의 고리 수’를 예측한 뒤, **예측된 ‘껍질의 고리 수’에 1.5를 더하면 전복의 나이**가 됩니다.

In [1]:
# 기본 모듈 불러오기
import numpy as np
import pandas as pd

**1) 데이터 load 및 변형**

In [2]:
# 데이터 로드
data = pd.read_csv("abalone.csv")
data.head()
print(data.shape)

# 성별 M은 Male, F는 Female, I는 Infant 이므로 따로 열 만들기
for label in "MFI":
    data[label] = data["Sex"] == label
data.drop('Sex', axis=1, inplace=True)

(4177, 9)


**2) X, y 선택**
: y는 Rings열, X는 Rings열을 제외한 나머지를 선택하되 전부 실수가 되도록 한다.

In [3]:
# X,y 데이터 선택
# y: 예측하고 싶은 값(껍질의 고리 수)
y = data['Rings']

# X: Rings를 제외한 나머지 열. M/F/I 열이 True/False(bool)이므로 astype(float)로 0.0/1.0 실수로 바꾼다
X = data.drop('Rings', axis=1).astype(float)

 **3) train/test set 분리**

In [4]:
# 필요한 모듈 불러오기
from sklearn.model_selection import train_test_split

In [5]:
# train과 test set 분리 (train:test = 7:3 비율로)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=50)

**4) 선형회귀모델 생성, 모델 예측치 구하기**

In [6]:
#필요한 모듈 불러오기
from sklearn.linear_model import LinearRegression

In [7]:
#선형회귀모델 생성 및 훈련
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [8]:
# 모델 예측치 구하기 (test set의 '껍질의 고리 수' 예측)
y_pred = lr.predict(X_test)

# 모델 예측치를 활용해 최종적으로 전복의 나이를 예측 (나이 = 고리 수 + 1.5)
age_pred = y_pred + 1.5
print(age_pred[:5])

[15.71298174 13.62942612  9.33871343 12.22531016  8.27725796]


**5) 모델 평가: MSE, RMSE, R2 score, corr 구하기**

In [9]:
#필요한 모듈 불러오기
from sklearn.metrics import mean_squared_error, r2_score

- MSE, RMSE

In [10]:
#mse, rmse
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print('MSE :', mse)
print('RMSE:', rmse)

MSE : 5.361803632532983
RMSE: 2.315556873094026


- R2 score

In [11]:
#R2 score 측정
r2 = r2_score(y_test, y_pred)
print('R2 score:', r2)

R2 score: 0.5200434489465241


- 회귀 절편값

In [12]:
#회귀 절편 값
print(lr.intercept_)

3.562307597974505


- 회귀 계수 값

In [13]:
#회귀 계수 값
# 어떤 변수의 계수인지 알 수 있도록 열 이름과 함께 출력
print(pd.Series(lr.coef_, index=X.columns))

Length            -1.143264
Diameter           8.931206
Height            21.586755
Whole weight       6.670507
Shucked weight   -16.818732
Viscera weight    -8.889032
Shell weight      10.156013
M                  0.320173
F                  0.286368
I                 -0.606541
dtype: float64


- 상관계수

Hint: corr 함수 이용.

In [14]:
# 상관계수 구하기 (실제값 y_test와 예측값 y_pred 사이의 상관계수)
y_pred_s = pd.Series(y_pred, index=y_test.index)   # corr은 Series끼리 인덱스가 맞아야 하므로 인덱스를 맞춰준다
print(y_test.corr(y_pred_s))

0.7211465887910177


# **2. Polynomial features**

In [15]:
# PolynomialFeatures 라이브러리 호출
from sklearn.preprocessing import PolynomialFeatures

In [16]:
# 임의 데이터 생성

X = np.arange(6).reshape(3, 2)

df =  pd.DataFrame(X)
df.columns = ['x_1','x_2']
df

,x_1,x_2
0,0,1
1,2,3
2,4,5


In [17]:
# 차원은 2로 설정
poly = PolynomialFeatures(degree=2)

# fit_transform 메소드를 통해 데이터 변환
poly_features = poly.fit_transform(df)

# PolynomialFeatures로 변환 된 데이터를 데이터 프레임 형태로 변환
df_poly = pd.DataFrame(poly_features)
df_poly

,0,1,2,3,4,5
0,1.0,0.0,1.0,0.0,0.0,1.0
1,1.0,2.0,3.0,4.0,6.0,9.0
2,1.0,4.0,5.0,16.0,20.0,25.0


In [18]:
# df_poly의 컬럼을 1,x1,x2,x1^2,x1*x2,x2^2 로 변경
df_poly.columns = ['1', 'x1', 'x2', 'x1^2', 'x1*x2', 'x2^2']
df_poly

,1,x1,x2,x1^2,x1*x2,x2^2
0,1.0,0.0,1.0,0.0,0.0,1.0
1,1.0,2.0,3.0,4.0,6.0,9.0
2,1.0,4.0,5.0,16.0,20.0,25.0
